In [128]:
import pandas as pd
import os
import re
import numpy as np

In [129]:
SCRIPT_DIR_PATH = os.getcwd()
CB_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
SSP_MODELING_DIR_PATH = os.path.dirname(CB_DIR_PATH)
TORNADO_DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
INPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "input/whirlpool")
OUTPUT_DATA_DIR_PATH = os.path.join(TORNADO_DATA_DIR_PATH, "output/whirlpool")

In [130]:
def add_sector_and_transformation_fields(df: pd.DataFrame, strategy_col: str = "strategy") -> pd.DataFrame:
    df = df.copy()

    df["sector"] = df[strategy_col].str.extract(r"TX:([A-Z]{3,6}):", expand=False)

    df.loc[df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False), "sector"] = "BASE"

    df["transformation_name"] = df[strategy_col].str.extract(r"TX:[A-Z]{3,6}:(\S+)", expand=False)

    base_mask = df[strategy_col].str.contains(r"TX:BASE", regex=True, na=False)
    df.loc[base_mask, "transformation_name"] = "BASE"

    # Elimina "_STRATEGY_XX" al final
    df["transformation_name"] = df["transformation_name"].str.replace(r"All Actions", "", regex=True)

    df["transformation_name"] = df["transformation_name"].fillna("").str.strip()

    return df

## Load and process emission data

In [131]:
# Load the decomposed emissions long format data
emissions_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "decomposed_emissions_libya_2022_whirlpool_data_raw.csv"))

emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
0,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.035964,2023,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
1,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.037243,2024,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
2,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.038469,2025,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
3,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039256,2026,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE
4,0.0,0.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039666,2027,CH4,0.0,0.0,Strategy TX:BASE,LBY,libya,SISEPUEDE


In [132]:
print(emissions_df.primary_id.nunique())

70


In [133]:
# check unique strategy
emissions_df['strategy'].unique()

array(['Strategy TX:BASE', 'All Actions',
       'Remove TX:AGRC:DEC_CH4_RICE from All Actions',
       'Remove TX:AGRC:DEC_EXPORTS from All Actions',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from All Actions',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE from All Actions',
       'Remove TX:AGRC:INC_PRODUCTIVITY from All Actions',
       'Remove TX:CCSQ:INC_CAPTURE from All Actions',
       'Remove TX:ENFU:ADJ_EXPORTS from All Actions',
       'Remove TX:ENTC:DEC_LOSSES from All Actions',
       'Remove TX:ENTC:LEAST_COST_SOLUTION from All Actions',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN from All Actions',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC from All Actions',
       'Remove TX:FGTV:DEC_LEAKS from All Actions',
       'Remove TX:FGTV:INC_FLARE from All Actions',
       'Remove TX:FRST:INCREASE_SEQUESTRATION from All Actions',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY from All Actions',
       'Remove TX:INEN:INC_EFFICIENCY_PRODUCTION from All Actions',
 

In [134]:
# Drop historical and tx:base from df
filtered_emissions_df = emissions_df.loc[~emissions_df['strategy'].isin(['Historical', 'Strategy TX:BASE'])]
print(emissions_df['strategy'].nunique())
print(filtered_emissions_df['strategy'].nunique())

71
69


In [135]:
filtered_emissions_df["strategy"].unique()

array(['All Actions', 'Remove TX:AGRC:DEC_CH4_RICE from All Actions',
       'Remove TX:AGRC:DEC_EXPORTS from All Actions',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from All Actions',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE from All Actions',
       'Remove TX:AGRC:INC_PRODUCTIVITY from All Actions',
       'Remove TX:CCSQ:INC_CAPTURE from All Actions',
       'Remove TX:ENFU:ADJ_EXPORTS from All Actions',
       'Remove TX:ENTC:DEC_LOSSES from All Actions',
       'Remove TX:ENTC:LEAST_COST_SOLUTION from All Actions',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN from All Actions',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC from All Actions',
       'Remove TX:FGTV:DEC_LEAKS from All Actions',
       'Remove TX:FGTV:INC_FLARE from All Actions',
       'Remove TX:FRST:INCREASE_SEQUESTRATION from All Actions',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY from All Actions',
       'Remove TX:INEN:INC_EFFICIENCY_PRODUCTION from All Actions',
       'Remove TX:INEN:SHIFT

In [136]:
filtered_emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
1148,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.035964,2023,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE
1149,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.037243,2024,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE
1150,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.038469,2025,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE
1151,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039256,2026,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE
1152,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039666,2027,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE


In [137]:
filtered_emissions_df.tail()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
80355,6072.0,143143.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.007005,2046,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING from All Actions,LBY,libya,SISEPUEDE
80356,6072.0,143143.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.007394,2047,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING from All Actions,LBY,libya,SISEPUEDE
80357,6072.0,143143.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.007783,2048,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING from All Actions,LBY,libya,SISEPUEDE
80358,6072.0,143143.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.008172,2049,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING from All Actions,LBY,libya,SISEPUEDE
80359,6072.0,143143.0,5- CCSQ:N2O,5- CCSQ,5- CCSQ,0.008561,2050,N2O,0.0,0.0,Remove TX:WASO:INC_RECYCLING from All Actions,LBY,libya,SISEPUEDE


In [138]:
# Now concat the original base df and the filtered emissions df
tornado_emissions_df = filtered_emissions_df
tornado_emissions_df['strategy'].unique()

array(['All Actions', 'Remove TX:AGRC:DEC_CH4_RICE from All Actions',
       'Remove TX:AGRC:DEC_EXPORTS from All Actions',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from All Actions',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE from All Actions',
       'Remove TX:AGRC:INC_PRODUCTIVITY from All Actions',
       'Remove TX:CCSQ:INC_CAPTURE from All Actions',
       'Remove TX:ENFU:ADJ_EXPORTS from All Actions',
       'Remove TX:ENTC:DEC_LOSSES from All Actions',
       'Remove TX:ENTC:LEAST_COST_SOLUTION from All Actions',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN from All Actions',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC from All Actions',
       'Remove TX:FGTV:DEC_LEAKS from All Actions',
       'Remove TX:FGTV:INC_FLARE from All Actions',
       'Remove TX:FRST:INCREASE_SEQUESTRATION from All Actions',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY from All Actions',
       'Remove TX:INEN:INC_EFFICIENCY_PRODUCTION from All Actions',
       'Remove TX:INEN:SHIFT

In [139]:
tornado_emissions_df.head()

,strategy_id,primary_id,ID,sector,subsector,value,Year,Gas,design_id,future_id,strategy,Code,Contry,source
1148,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.035964,2023,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE
1149,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.037243,2024,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE
1150,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.038469,2025,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE
1151,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039256,2026,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE
1152,6002.0,73073.0,1.A.1 - Energy Industries:CH4,1 - Energy,1.A.1 - Energy Industries,0.039666,2027,CH4,0.0,0.0,All Actions,LBY,libya,SISEPUEDE


In [140]:
# Aggregate by strategy_id, primary_id and strategy, and sum value
tornado_emissions_agg_df = tornado_emissions_df.groupby(
    ['strategy_id', 'primary_id', 'strategy']
)['value'].sum().reset_index()

tornado_emissions_agg_df.head()


,strategy_id,primary_id,strategy,value
0,6002.0,73073.0,All Actions,1251.819354
1,6005.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE from All Actions,1251.819354
2,6006.0,77077.0,Remove TX:AGRC:DEC_EXPORTS from All Actions,1251.819354
3,6007.0,78078.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from Al...,1251.973079
4,6008.0,79079.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE fr...,1250.776119


In [141]:
tornado_emissions_agg_df.tail()

,strategy_id,primary_id,strategy,value
64,6068.0,139139.0,Remove TX:WASO:INC_CAPTURE_BIOGAS from All Act...,1251.801645
65,6069.0,140140.0,Remove TX:WASO:INC_ENERGY_FROM_BIOGAS from All...,1251.783771
66,6070.0,141141.0,Remove TX:WASO:INC_ENERGY_FROM_INCINERATION fr...,1252.021943
67,6071.0,142142.0,Remove TX:WASO:INC_LANDFILLING from All Actions,1253.461057
68,6072.0,143143.0,Remove TX:WASO:INC_RECYCLING from All Actions,1256.091107


In [142]:
# check if strategy id nunique matches amount of rows
print(tornado_emissions_agg_df['strategy_id'].nunique())
print(tornado_emissions_agg_df.shape[0])

69
69


In [143]:
# rename value to emission_total
tornado_emissions_agg_df = tornado_emissions_agg_df.rename(columns={'value': 'emission_total'})

# create base_emission_total column by setting it to the strategy_id == 0 value
base_emission_total = (tornado_emissions_agg_df.loc[tornado_emissions_agg_df['strategy_id'] == 6002, 'emission_total'].values[0]) 
tornado_emissions_agg_df['base_emission_total'] = base_emission_total 

base_emission_total

np.float64(1251.819353628363)

In [144]:
# calculate emission difference column
tornado_emissions_agg_df['emission_diff'] =  tornado_emissions_agg_df['emission_total'] - tornado_emissions_agg_df['base_emission_total']

tornado_emissions_agg_df['emission_diff'] = tornado_emissions_agg_df['emission_diff'].round(1)

tornado_emissions_agg_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff
0,6002.0,73073.0,All Actions,1251.819354,1251.819354,0.0
1,6005.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE from All Actions,1251.819354,1251.819354,0.0
2,6006.0,77077.0,Remove TX:AGRC:DEC_EXPORTS from All Actions,1251.819354,1251.819354,0.0
3,6007.0,78078.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from Al...,1251.973079,1251.819354,0.2
4,6008.0,79079.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE fr...,1250.776119,1251.819354,-1.0


In [145]:
tornado_emissions_agg_extended_df = add_sector_and_transformation_fields(tornado_emissions_agg_df)
tornado_emissions_agg_extended_df.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
0,6002.0,73073.0,All Actions,1251.819354,1251.819354,0.0,NaN,
1,6005.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_CH4_RICE
2,6006.0,77077.0,Remove TX:AGRC:DEC_EXPORTS from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_EXPORTS
3,6007.0,78078.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from Al...,1251.973079,1251.819354,0.2,AGRC,DEC_LOSSES_SUPPLY_CHAIN
4,6008.0,79079.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE fr...,1250.776119,1251.819354,-1.0,AGRC,INC_CONSERVATION_AGRICULTURE


In [146]:
tornado_emissions_agg_extended_df.to_clipboard(index=False)

## Load and process CB data

In [147]:
# cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "costs_benefits_sisepuede_results_sisepuede_run_2026-01-29T15;28;40.322709_tornado_raw.csv"))
cb_raw_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "cba_results_ssp_modeling_whirlpool.csv"))
cb_raw_df.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value
0,PFLO:ALL,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0
1,PFLO:ALL,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0
2,PFLO:ALL,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0
3,PFLO:ALL,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0
4,PFLO:ALL,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0


In [148]:
# --- Create a copy of the raw data ---
cb_data = cb_raw_df.copy()

# Split 'variable' into components: name, sector, cb_type, item_1, item_2
# (Assumes exactly 5 colon-separated parts; if there are more colons inside the last field,
# they will be kept in item_2 thanks to n=4)
cb_chars = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
cb_chars.columns = ["name", "sector", "cb_type", "item_1", "item_2"]
cb_data = pd.concat([cb_data, cb_chars], axis=1)
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2
0,PFLO:ALL,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices
1,PFLO:ALL,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices
2,PFLO:ALL,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices
3,PFLO:ALL,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices
4,PFLO:ALL,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices


In [149]:
# Scale value from USD to billions (divide by 1e9)
if "value" in cb_data.columns:
    cb_data["value"] = cb_data["value"] / 1e9

# --- Remove "shifted" entries ---
# # Remove rows where item_2 contains "shifted"
# cb_data = cb_data[~cb_data["item_2"].astype(str).str.contains("shifted", na=False)]

# # Remove any remaining rows where variable contains "shifted2"
# cb_data = cb_data[~cb_data["variable"].astype(str).str.contains("shifted2", na=False)]

# --- Add Year column (Year = time_period + 2015) ---
cb_data["Year"] = cb_data["time_period"] + 2015

cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year
0,PFLO:ALL,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2022.0
1,PFLO:ALL,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2023.0
2,PFLO:ALL,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2024.0
3,PFLO:ALL,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2025.0
4,PFLO:ALL,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2026.0


In [150]:
# Load attribute strategy
attribute_strategy_df = pd.read_csv(os.path.join(INPUT_DATA_DIR_PATH, "ATTRIBUTE_STRATEGY.csv"))
attribute_strategy_df = attribute_strategy_df[["strategy_id", "strategy_code"]]
attribute_strategy_df.head()

,strategy_id,strategy_code
0,0,BASE
1,6002,PFLO:ALL
2,6005,WHIRLPOOL:TX:AGRC:DEC_CH4_RICE
3,6006,WHIRLPOOL:TX:AGRC:DEC_EXPORTS
4,6007,WHIRLPOOL:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN


In [151]:
attribute_strategy_df.strategy_id.unique()

array([   0, 6002, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6012, 6013,
       6014, 6015, 6016, 6017, 6018, 6019, 6020, 6021, 6022, 6023, 6024,
       6025, 6026, 6027, 6028, 6029, 6030, 6031, 6032, 6033, 6034, 6035,
       6036, 6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045, 6046,
       6047, 6048, 6049, 6050, 6051, 6052, 6053, 6054, 6055, 6056, 6057,
       6058, 6059, 6060, 6061, 6062, 6063, 6064, 6065, 6066, 6067, 6068,
       6069, 6070, 6071, 6072])

In [152]:
attribute_strategy_df.strategy_code.unique()

array(['BASE', 'PFLO:ALL', 'WHIRLPOOL:TX:AGRC:DEC_CH4_RICE',
       'WHIRLPOOL:TX:AGRC:DEC_EXPORTS',
       'WHIRLPOOL:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN',
       'WHIRLPOOL:TX:AGRC:INC_CONSERVATION_AGRICULTURE',
       'WHIRLPOOL:TX:AGRC:INC_PRODUCTIVITY',
       'WHIRLPOOL:TX:CCSQ:INC_CAPTURE', 'WHIRLPOOL:TX:ENFU:ADJ_EXPORTS',
       'WHIRLPOOL:TX:ENTC:DEC_LOSSES',
       'WHIRLPOOL:TX:ENTC:LEAST_COST_SOLUTION',
       'WHIRLPOOL:TX:ENTC:TARGET_CLEAN_HYDROGEN',
       'WHIRLPOOL:TX:ENTC:TARGET_RENEWABLE_ELEC',
       'WHIRLPOOL:TX:FGTV:DEC_LEAKS', 'WHIRLPOOL:TX:FGTV:INC_FLARE',
       'WHIRLPOOL:TX:FRST:INCREASE_SEQUESTRATION',
       'WHIRLPOOL:TX:INEN:INC_EFFICIENCY_ENERGY',
       'WHIRLPOOL:TX:INEN:INC_EFFICIENCY_PRODUCTION',
       'WHIRLPOOL:TX:INEN:SHIFT_FUEL_HEAT',
       'WHIRLPOOL:TX:IPPU:DEC_CLINKER', 'WHIRLPOOL:TX:IPPU:DEC_DEMAND',
       'WHIRLPOOL:TX:IPPU:DEC_HFCS', 'WHIRLPOOL:TX:IPPU:DEC_N2O',
       'WHIRLPOOL:TX:IPPU:DEC_OTHER_FCS', 'WHIRLPOOL:TX:IPPU:DEC_PFCS',
       

In [153]:
# Merge with cb_data on strategy_code
cb_data = cb_data.merge(attribute_strategy_df, on="strategy_code", how="left")
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:ALL,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2022.0,6002
1,PFLO:ALL,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,6002
2,PFLO:ALL,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,6002
3,PFLO:ALL,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,6002
4,PFLO:ALL,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,6002


In [154]:
cb_data.strategy_code.unique()

array(['PFLO:ALL', 'WHIRLPOOL:TX:AGRC:DEC_CH4_RICE',
       'WHIRLPOOL:TX:AGRC:DEC_EXPORTS',
       'WHIRLPOOL:TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN',
       'WHIRLPOOL:TX:AGRC:INC_CONSERVATION_AGRICULTURE',
       'WHIRLPOOL:TX:AGRC:INC_PRODUCTIVITY',
       'WHIRLPOOL:TX:CCSQ:INC_CAPTURE', 'WHIRLPOOL:TX:ENFU:ADJ_EXPORTS',
       'WHIRLPOOL:TX:ENTC:DEC_LOSSES',
       'WHIRLPOOL:TX:ENTC:LEAST_COST_SOLUTION',
       'WHIRLPOOL:TX:ENTC:TARGET_CLEAN_HYDROGEN',
       'WHIRLPOOL:TX:ENTC:TARGET_RENEWABLE_ELEC',
       'WHIRLPOOL:TX:FGTV:DEC_LEAKS', 'WHIRLPOOL:TX:FGTV:INC_FLARE',
       'WHIRLPOOL:TX:FRST:INCREASE_SEQUESTRATION',
       'WHIRLPOOL:TX:INEN:INC_EFFICIENCY_ENERGY',
       'WHIRLPOOL:TX:INEN:INC_EFFICIENCY_PRODUCTION',
       'WHIRLPOOL:TX:INEN:SHIFT_FUEL_HEAT',
       'WHIRLPOOL:TX:IPPU:DEC_CLINKER', 'WHIRLPOOL:TX:IPPU:DEC_DEMAND',
       'WHIRLPOOL:TX:IPPU:DEC_HFCS', 'WHIRLPOOL:TX:IPPU:DEC_N2O',
       'WHIRLPOOL:TX:IPPU:DEC_OTHER_FCS', 'WHIRLPOOL:TX:IPPU:DEC_PFCS',
       'WHIRLPO

In [155]:
cb_data.strategy_id.unique()

array([6002, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6012, 6013, 6014,
       6015, 6016, 6017, 6018, 6019, 6020, 6021, 6022, 6023, 6024, 6025,
       6026, 6027, 6028, 6029, 6030, 6031, 6032, 6033, 6034, 6035, 6036,
       6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045, 6046, 6047,
       6048, 6049, 6050, 6051, 6052, 6053, 6054, 6055, 6056, 6057, 6058,
       6059, 6060, 6061, 6062, 6063, 6064, 6065, 6066, 6067, 6068, 6069,
       6070, 6071, 6072])

In [156]:
# check for nans in strategy_id
cb_data[cb_data['strategy_id'].isna()]['strategy_code'].unique()

array([], dtype=object)

In [157]:
cb_data["sector"].unique()

array(['agrc', 'ccsq', 'entc', 'inen', 'ippu', 'lndu', 'lsmm', 'lvst',
       'scoe', 'soil', 'trns', 'trww', 'wali', 'waso', 'fgtv', 'pflo'],
      dtype=object)

In [158]:
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:ALL,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2022.0,6002
1,PFLO:ALL,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,6002
2,PFLO:ALL,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,6002
3,PFLO:ALL,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,6002
4,PFLO:ALL,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,6002


In [159]:
# filter sectors
# target_sectors = ["wali", "trww", "waso", "soil", "ippu", "lvst", "agrc", "lndu", "lsmm"]
# cb_data = cb_data[cb_data["sector"].isin(target_sectors)].copy()
cb_data.head()

,strategy_code,future_id,region,time_period,difference_variable,variable_value_baseline,variable_value_pathway,difference_value,variable,value,name,sector,cb_type,item_1,item_2,Year,strategy_id
0,PFLO:ALL,0.0,libya,7.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2022.0,6002
1,PFLO:ALL,0.0,libya,8.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2023.0,6002
2,PFLO:ALL,0.0,libya,9.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2024.0,6002
3,PFLO:ALL,0.0,libya,10.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2025.0,6002
4,PFLO:ALL,0.0,libya,11.0,yield_agrc_bevs_and_spices_tonne,0.0,0.0,0.0,cb:agrc:crop_value:crops_produced:bevs_and_spices,0.0,cb,agrc,crop_value,crops_produced,bevs_and_spices,2026.0,6002


In [160]:
# aggregate sum(value) grouped by strategy_id and cb_type
cb_data = (
    cb_data.groupby(["strategy_id", "cb_type"], as_index=False)["value"]
      .sum()
      .rename(columns={"value": "cumulative"})
)
cb_data.head(20)

,strategy_id,cb_type,cumulative
0,6002,air_pollution,33.676212
1,6002,congestion,17.519533
2,6002,consumer_savings,30.748214
3,6002,crop_value,-114.095128
4,6002,ecosystem_services,5.777444
5,6002,env_pollution,12.637657
6,6002,fuel_cost,5.683741
7,6002,human_health,32.424529
8,6002,ippu_value,2.240758
9,6002,land_pollution,0.034157


In [161]:
# unique cb_data types
cb_cats = cb_data["cb_type"].unique().tolist()

# long -> wide (R dcast equivalent)
wide_cb = (
    cb_data.pivot(index="strategy_id", columns="cb_type", values="cumulative")
      .reset_index()
)

# optional: remove column name from pivot for nicer printing
wide_cb.columns.name = None
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,6002,33.676212,17.519533,30.748214,-114.095128,5.777444,12.637657,5.683741,32.424529,2.240758,0.034157,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499
1,6005,33.676212,17.519533,30.748214,-114.095128,5.777444,12.637657,5.683741,32.424529,2.240758,0.034157,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499
2,6006,33.676212,17.519533,30.748214,-114.095128,5.777444,12.637657,5.683741,32.424529,2.240758,0.034157,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499
3,6007,33.673436,17.519533,31.075706,-111.365127,5.710261,12.623604,5.707195,32.424529,2.240758,0.033732,2.026626,26.560288,122.899466,215.776864,-243.133339,-181.222903,14.627498
4,6008,33.676212,17.519533,20.485666,-114.095128,5.777444,12.637657,5.683741,32.424529,2.240758,0.034157,2.026592,26.560288,122.945120,215.776864,-242.843087,-179.426547,14.627499


In [162]:
cb_cats

['air_pollution',
 'congestion',
 'consumer_savings',
 'crop_value',
 'ecosystem_services',
 'env_pollution',
 'fuel_cost',
 'human_health',
 'ippu_value',
 'land_pollution',
 'lvst_value',
 'road_safety',
 'sector_specific',
 'system_cost',
 'technical_cost',
 'technical_savings',
 'water_pollution']

In [163]:
# --- 1) net_benefit = rowSums over all cb categories ---
wide_cb["net_benefit"] = wide_cb[cb_cats].sum(axis=1, skipna=True)

# --- 2) additional_benefits = rowSums excluding "technical_cost" ---
benefit_cols = [c for c in cb_cats if c != "technical_cost"]
wide_cb["additional_benefits"] = wide_cb[benefit_cols].sum(axis=1, skipna=True)

# --- 3) total_transformation_costs = rowSums over specific cols ---
cost_cols = ["technical_cost", "technical_savings", "fuel_cost"]

# (safe version: only use cols that exist in the df)
cost_cols = [c for c in cost_cols if c in wide_cb.columns]

wide_cb["total_transformation_costs"] = wide_cb[cost_cols].sum(axis=1, skipna=True)
wide_cb.head()

,strategy_id,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,fuel_cost,human_health,ippu_value,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6002,33.676212,17.519533,30.748214,-114.095128,5.777444,12.637657,5.683741,32.424529,2.240758,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499,-12.483971,230.359116,-415.383709
1,6005,33.676212,17.519533,30.748214,-114.095128,5.777444,12.637657,5.683741,32.424529,2.240758,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499,-12.483971,230.359116,-415.383709
2,6006,33.676212,17.519533,30.748214,-114.095128,5.777444,12.637657,5.683741,32.424529,2.240758,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499,-12.483971,230.359116,-415.383709
3,6007,33.673436,17.519533,31.075706,-111.365127,5.710261,12.623604,5.707195,32.424529,2.240758,...,2.026626,26.560288,122.899466,215.776864,-243.133339,-181.222903,14.627498,-12.821871,230.311468,-418.649046
4,6008,33.676212,17.519533,20.485666,-114.095128,5.777444,12.637657,5.683741,32.424529,2.240758,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-179.426547,14.627499,-23.948703,218.894384,-416.585893


## Merge emissions and cb data and save

In [164]:
tornado_emissions_agg_extended_df[tornado_emissions_agg_extended_df.strategy == "Remove TX:AGRC:DEC_CH4_RICE from All Actions"]

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name
1,6005.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_CH4_RICE


In [165]:
wide_cb.strategy_id.unique()

array([6002, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6012, 6013, 6014,
       6015, 6016, 6017, 6018, 6019, 6020, 6021, 6022, 6023, 6024, 6025,
       6026, 6027, 6028, 6029, 6030, 6031, 6032, 6033, 6034, 6035, 6036,
       6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045, 6046, 6047,
       6048, 6049, 6050, 6051, 6052, 6053, 6054, 6055, 6056, 6057, 6058,
       6059, 6060, 6061, 6062, 6063, 6064, 6065, 6066, 6067, 6068, 6069,
       6070, 6071, 6072])

In [166]:
print(wide_cb.shape)
print(tornado_emissions_agg_extended_df.shape)

(69, 21)
(69, 8)


In [167]:
df_merged = pd.merge(
    tornado_emissions_agg_extended_df,
    wide_cb,
    on="strategy_id",
    how="inner"
)

df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
0,6002.0,73073.0,All Actions,1251.819354,1251.819354,0.0,NaN,,33.676212,17.519533,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499,-12.483971,230.359116,-415.383709
1,6005.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_CH4_RICE,33.676212,17.519533,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499,-12.483971,230.359116,-415.383709
2,6006.0,77077.0,Remove TX:AGRC:DEC_EXPORTS from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_EXPORTS,33.676212,17.519533,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499,-12.483971,230.359116,-415.383709
3,6007.0,78078.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from Al...,1251.973079,1251.819354,0.2,AGRC,DEC_LOSSES_SUPPLY_CHAIN,33.673436,17.519533,...,2.026626,26.560288,122.899466,215.776864,-243.133339,-181.222903,14.627498,-12.821871,230.311468,-418.649046
4,6008.0,79079.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE fr...,1250.776119,1251.819354,-1.0,AGRC,INC_CONSERVATION_AGRICULTURE,33.676212,17.519533,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-179.426547,14.627499,-23.948703,218.894384,-416.585893


In [168]:
df_merged = df_merged[df_merged['strategy'] != 'All Actions']

df_merged.head()


,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs
1,6005.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_CH4_RICE,33.676212,17.519533,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499,-12.483971,230.359116,-415.383709
2,6006.0,77077.0,Remove TX:AGRC:DEC_EXPORTS from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_EXPORTS,33.676212,17.519533,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-178.224363,14.627499,-12.483971,230.359116,-415.383709
3,6007.0,78078.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from Al...,1251.973079,1251.819354,0.2,AGRC,DEC_LOSSES_SUPPLY_CHAIN,33.673436,17.519533,...,2.026626,26.560288,122.899466,215.776864,-243.133339,-181.222903,14.627498,-12.821871,230.311468,-418.649046
4,6008.0,79079.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE fr...,1250.776119,1251.819354,-1.0,AGRC,INC_CONSERVATION_AGRICULTURE,33.676212,17.519533,...,2.026592,26.560288,122.945120,215.776864,-242.843087,-179.426547,14.627499,-23.948703,218.894384,-416.585893
5,6009.0,80080.0,Remove TX:AGRC:INC_PRODUCTIVITY from All Actions,1251.056132,1251.819354,-0.8,AGRC,INC_PRODUCTIVITY,33.710669,17.519533,...,2.026690,26.560288,123.471975,215.776864,-242.695965,-178.382743,14.627499,-28.402776,214.293188,-415.394956


In [169]:
print(df_merged.shape)

(68, 28)


### Below we have some hardcoded fixed exclusive of this study case to replace incorrect tranformation names

In [170]:
df_merged.columns

Index(['strategy_id', 'primary_id', 'strategy', 'emission_total',
       'base_emission_total', 'emission_diff', 'sector', 'transformation_name',
       'air_pollution', 'congestion', 'consumer_savings', 'crop_value',
       'ecosystem_services', 'env_pollution', 'fuel_cost', 'human_health',
       'ippu_value', 'land_pollution', 'lvst_value', 'road_safety',
       'sector_specific', 'system_cost', 'technical_cost', 'technical_savings',
       'water_pollution', 'net_benefit', 'additional_benefits',
       'total_transformation_costs'],
      dtype='object')

In [171]:
# Get the base technical cost from wide_cb (strategy_id 6004)
base_technical_cost = (wide_cb[wide_cb['strategy_id'] == 6002]['technical_cost'].values[0])*-1
base_technical_cost

np.float64(242.8430872354803)

In [172]:
# multiply technical_cost by -1 to get positive costs
df_merged['technical_cost'] = df_merged['technical_cost'] * -1

df_merged['technical_cost'] =  df_merged['technical_cost'] - base_technical_cost

# create marginal total abatement cost column
df_merged['marginal_total_abatement_cost_(USD/tCO2e)'] = ((df_merged['technical_cost']*1e9) / (df_merged['emission_diff']*1e6))

# If technical_cost is positive then marginal_total_abatement_cost should be positive too.
df_merged["marginal_total_abatement_cost_(USD/tCO2e)"] = df_merged["marginal_total_abatement_cost_(USD/tCO2e)"].abs() * np.sign(df_merged["technical_cost"])

In [173]:
df_merged[df_merged['strategy_id'] == 6023]

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e)
19,6023.0,94094.0,Remove TX:IPPU:DEC_DEMAND from All Actions,1266.087492,1251.819354,14.3,IPPU,DEC_DEMAND,32.919499,17.519533,...,26.560288,116.677663,215.776864,2.881629,-175.324616,14.627499,-21.18937,224.535346,-415.36559,201.512504


In [174]:
df_merged["strategy"].unique()

array(['Remove TX:AGRC:DEC_CH4_RICE from All Actions',
       'Remove TX:AGRC:DEC_EXPORTS from All Actions',
       'Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from All Actions',
       'Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE from All Actions',
       'Remove TX:AGRC:INC_PRODUCTIVITY from All Actions',
       'Remove TX:CCSQ:INC_CAPTURE from All Actions',
       'Remove TX:ENFU:ADJ_EXPORTS from All Actions',
       'Remove TX:ENTC:DEC_LOSSES from All Actions',
       'Remove TX:ENTC:LEAST_COST_SOLUTION from All Actions',
       'Remove TX:ENTC:TARGET_CLEAN_HYDROGEN from All Actions',
       'Remove TX:ENTC:TARGET_RENEWABLE_ELEC from All Actions',
       'Remove TX:FGTV:DEC_LEAKS from All Actions',
       'Remove TX:FGTV:INC_FLARE from All Actions',
       'Remove TX:FRST:INCREASE_SEQUESTRATION from All Actions',
       'Remove TX:INEN:INC_EFFICIENCY_ENERGY from All Actions',
       'Remove TX:INEN:INC_EFFICIENCY_PRODUCTION from All Actions',
       'Remove TX:INEN:SHIFT_FUEL_HEAT from

In [175]:
df_merged['transformation_name_sector'] = df_merged['transformation_name'] + " - " + df_merged['sector']

In [176]:
df_merged.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_whirlpool.csv"), index=False)

### Create a QA version

In [177]:
df_merged.head()

,strategy_id,primary_id,strategy,emission_total,base_emission_total,emission_diff,sector,transformation_name,air_pollution,congestion,...,sector_specific,system_cost,technical_cost,technical_savings,water_pollution,net_benefit,additional_benefits,total_transformation_costs,marginal_total_abatement_cost_(USD/tCO2e),transformation_name_sector
1,6005.0,76076.0,Remove TX:AGRC:DEC_CH4_RICE from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_CH4_RICE,33.676212,17.519533,...,122.945120,215.776864,0.000000,-178.224363,14.627499,-12.483971,230.359116,-415.383709,NaN,DEC_CH4_RICE - AGRC
2,6006.0,77077.0,Remove TX:AGRC:DEC_EXPORTS from All Actions,1251.819354,1251.819354,0.0,AGRC,DEC_EXPORTS,33.676212,17.519533,...,122.945120,215.776864,0.000000,-178.224363,14.627499,-12.483971,230.359116,-415.383709,NaN,DEC_EXPORTS - AGRC
3,6007.0,78078.0,Remove TX:AGRC:DEC_LOSSES_SUPPLY_CHAIN from Al...,1251.973079,1251.819354,0.2,AGRC,DEC_LOSSES_SUPPLY_CHAIN,33.673436,17.519533,...,122.899466,215.776864,0.290252,-181.222903,14.627498,-12.821871,230.311468,-418.649046,1451.257980,DEC_LOSSES_SUPPLY_CHAIN - AGRC
4,6008.0,79079.0,Remove TX:AGRC:INC_CONSERVATION_AGRICULTURE fr...,1250.776119,1251.819354,-1.0,AGRC,INC_CONSERVATION_AGRICULTURE,33.676212,17.519533,...,122.945120,215.776864,0.000000,-179.426547,14.627499,-23.948703,218.894384,-416.585893,0.000000,INC_CONSERVATION_AGRICULTURE - AGRC
5,6009.0,80080.0,Remove TX:AGRC:INC_PRODUCTIVITY from All Actions,1251.056132,1251.819354,-0.8,AGRC,INC_PRODUCTIVITY,33.710669,17.519533,...,123.471975,215.776864,-0.147123,-178.382743,14.627499,-28.402776,214.293188,-415.394956,-183.903326,INC_PRODUCTIVITY - AGRC


In [178]:
df_merged.sector.unique()

array(['AGRC', 'CCSQ', 'ENFU', 'ENTC', 'FGTV', 'FRST', 'INEN', 'IPPU',
       'LNDU', 'LSMM', 'LVST', 'PFLO', 'SCOE', 'SOIL', 'TRDE', 'TRNS',
       'TRWW', 'WALI', 'WASO'], dtype=object)

In [179]:
relevant_fields = [
    "transformation_name",
    "sector",
    "base_emission_total",
    "emission_total",
    "emission_diff",
    "technical_cost",
    "marginal_total_abatement_cost_(USD/tCO2e)"
]

# keep only relevant fields
df_merged_filtered = df_merged[relevant_fields]
df_merged_filtered.head()

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,DEC_CH4_RICE,AGRC,1251.819354,1251.819354,0.0,0.000000,NaN
2,DEC_EXPORTS,AGRC,1251.819354,1251.819354,0.0,0.000000,NaN
3,DEC_LOSSES_SUPPLY_CHAIN,AGRC,1251.819354,1251.973079,0.2,0.290252,1451.257980
4,INC_CONSERVATION_AGRICULTURE,AGRC,1251.819354,1250.776119,-1.0,0.000000,0.000000
5,INC_PRODUCTIVITY,AGRC,1251.819354,1251.056132,-0.8,-0.147123,-183.903326


In [180]:
df_merged_filtered

,transformation_name,sector,base_emission_total,emission_total,emission_diff,technical_cost,marginal_total_abatement_cost_(USD/tCO2e)
1,DEC_CH4_RICE,AGRC,1251.819354,1251.819354,0.0,0.000000,NaN
2,DEC_EXPORTS,AGRC,1251.819354,1251.819354,0.0,0.000000,NaN
3,DEC_LOSSES_SUPPLY_CHAIN,AGRC,1251.819354,1251.973079,0.2,0.290252,1451.257980
4,INC_CONSERVATION_AGRICULTURE,AGRC,1251.819354,1250.776119,-1.0,0.000000,0.000000
5,INC_PRODUCTIVITY,AGRC,1251.819354,1251.056132,-0.8,-0.147123,-183.903326
...,...,...,...,...,...,...,...
64,INC_CAPTURE_BIOGAS,WASO,1251.819354,1251.801645,-0.0,-0.279074,-inf
65,INC_ENERGY_FROM_BIOGAS,WASO,1251.819354,1251.783771,-0.0,-0.567780,-inf
66,INC_ENERGY_FROM_INCINERATION,WASO,1251.819354,1252.021943,0.2,1.921748,9608.738706
67,INC_LANDFILLING,WASO,1251.819354,1253.461057,1.6,-6.305108,-3940.692638


In [181]:
df_merged_filtered.to_clipboard(index=False)

In [182]:
df_merged_filtered.to_csv(os.path.join(OUTPUT_DATA_DIR_PATH, "tornado_plot_for_QA_whirlpool.csv"), index=False)